In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
# -------------------------
# Base Generators (Single Channel)
# -------------------------

def generate_trend_channel(T):
    t = np.linspace(0, 1, T)
    a = np.random.uniform(1.0, 3.0)
    b = np.random.uniform(-1.0, 1.0)
    return a * t**2 + b * t + np.random.normal(0, 0.05, T)


def generate_sine_channel(T):
    t = np.linspace(0, 1, T)
    freq = np.random.uniform(2, 5)
    phase = np.random.uniform(0, 2*np.pi)
    return np.sin(2 * np.pi * freq * t + phase) + np.random.normal(0, 0.05, T)


def generate_ar_channel(T):
    X = np.zeros(T)
    phi = np.random.uniform(0.6, 0.9)
    noise = np.random.normal(0, 0.1, T)
    
    for t in range(1, T):
        X[t] = phi * X[t-1] + noise[t]
    
    return X


# -------------------------
# Anomaly Injection (per channel type)
# -------------------------

def inject_trend_anomaly_channel(x, start, length):
    for t in range(start, start + length):
        delta = x[t] - x[start]
        x[t] -= (1.25 * delta**2) + 0.5 * delta
    return x


def inject_sine_anomaly_channel(x, start, length):
    t_local = np.arange(length)
    distorted = np.sin(5 * t_local / length)
    x[start:start+length] = distorted + np.random.normal(0, 0.05, length)
    return x


def inject_ar_anomaly_channel(x, start, length):
    noise = np.random.normal(0, 0.25, length)
    x[start:start+length] = noise
    return x


# -------------------------
# Main Dataset Generator
# -------------------------

def generate_dataset(T=200000, d=15, anomaly_length=25):
    
    X = np.zeros((T, d))
    y = np.zeros(T)
    
    # Split channels into thirds
    d1 = d // 3
    d2 = d // 3
    d3 = d - d1 - d2  # handle remainder cleanly
    
    # Track channel types (VERY useful for analysis later)
    channel_types = []
    
    # Generate channels
    for j in range(d):
        if j < d1:
            X[:, j] = generate_trend_channel(T)
            channel_types.append("trend")
        elif j < d1 + d2:
            X[:, j] = generate_sine_channel(T)
            channel_types.append("sine")
        else:
            X[:, j] = generate_ar_channel(T)
            channel_types.append("ar")
    
    # -------------------------
    # Inject anomalies
    # -------------------------
    
    start = np.random.randint(140000, 150000)
    
    for j in range(d):
        if channel_types[j] == "trend":
            X[:, j] = inject_trend_anomaly_channel(X[:, j], start, anomaly_length)
        elif channel_types[j] == "sine":
            X[:, j] = inject_sine_anomaly_channel(X[:, j], start, anomaly_length)
        else:
            X[:, j] = inject_ar_anomaly_channel(X[:, j], start, anomaly_length)
    
    y[start:start+anomaly_length] = 1
    
    return X, y, start